In [1]:
from autocvd import autocvd
autocvd(num_gpus = 1)

backend = "torch"
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
if "KERAS_BACKEND" not in os.environ:
    os.environ["KERAS_BACKEND"] = backend

import bayesflow as bf
import numpy as np
import keras

INFO:bayesflow:Using backend 'torch'
When using torch backend, we need to disable autograd by default to avoid excessive memory usage. Use

with torch.enable_grad():
    ...

in contexts where you need gradients (e.g. custom training loops).
/export/home/vgiusepp/miniconda3/envs/bf/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load training data and take only first 100 samples
training_data_raw = dict(np.load('./projection_training_set_odisseo_triaxial.npz', allow_pickle=True))

# Take first 100 samples
N_MOCK = 100
training_data = {k: v[:N_MOCK] for k, v in training_data_raw.items() 
                 if hasattr(v, 'shape') and len(v.shape) > 0 and v.shape[0] >= N_MOCK}

print("Mock training data shapes:")
for k, v in training_data.items():
    if hasattr(v, 'shape'):
        print(f"  {k}: {v.shape}")

Mock training data shapes:
  j: (100, 1)
  sim_data: (100, 1000, 6)
  m_nfw: (100, 1)
  r_s: (100, 1)
  q1: (100, 1)
  q2: (100, 1)
  prog_mass: (100, 1)
  t_end: (100, 1)
  x_c: (100, 1)
  y_c: (100, 1)
  z_c: (100, 1)
  v_xc: (100, 1)
  v_yc: (100, 1)
  v_zc: (100, 1)


In [3]:
# Get dimensions from data
n_samples = training_data['sim_data'].shape[0]  # 100
n_particles = training_data['sim_data'].shape[1]  # number of particles per sample

print(f"Number of samples: {n_samples}")
print(f"Number of particles per sample: {n_particles}")

# Create random attention masks with variable number of valid particles
# For each sample, randomly decide how many particles are "valid" (between 50% and 100%)
np.random.seed(42)

min_valid_fraction = 0.5
max_valid_fraction = 1.0

attention_masks = np.zeros((n_samples, n_particles), dtype=np.float32)

for i in range(n_samples):
    # Random fraction of valid particles for this sample
    valid_fraction = np.random.uniform(min_valid_fraction, max_valid_fraction)
    n_valid = int(n_particles * valid_fraction)
    
    # Randomly select which particles are valid
    valid_indices = np.random.choice(n_particles, size=n_valid, replace=False)
    attention_masks[i, valid_indices] = 1.0

print(f"Attention mask shape: {attention_masks.shape}")
print(f"Example valid counts per sample (first 10): {attention_masks[:10].sum(axis=1)}")
print(f"Min valid particles: {attention_masks.sum(axis=1).min()}")
print(f"Max valid particles: {attention_masks.sum(axis=1).max()}")

Number of samples: 100
Number of particles per sample: 1000
Attention mask shape: (100, 1000)
Example valid counts per sample (first 10): [687. 994. 689. 685. 818. 727. 657. 962. 998. 597.]
Min valid particles: 504.0
Max valid particles: 998.0


In [4]:
# Define parameter names
param_names_global = ['m_nfw', 'r_s', 'q1', 'q2']

# Create adapter
adapter = (
    bf.adapters.Adapter()
    .to_array()
    .convert_dtype("float64", "float32")
    .concatenate(param_names_global, into="inference_variables")
    .rename("j", "inference_conditions")
    .rename("sim_data", "summary_variables")
)

# Create a simple workflow for testing
workflow_test = bf.BasicWorkflow(
    adapter=adapter,
    summary_network=bf.networks.SetTransformer(summary_dim=16, dropout=0.1),
    inference_network=bf.networks.CompositionalDiffusionModel(),
    standardize=["inference_variables", "summary_variables"]
)

In [5]:
# Test 1: Check if SetTransformer accepts attention_mask in its call method
import inspect

print("SetTransformer.call signature:")
print(inspect.signature(workflow_test.approximator.summary_network.call))

# Check the attention_blocks
print("\nAttention blocks type:", type(workflow_test.approximator.summary_network.attention_blocks))

SetTransformer.call signature:
(input_set: ~Tensor, training: bool = False, **kwargs) -> ~Tensor

Attention blocks type: <class 'keras.src.models.sequential.Sequential'>


In [6]:
attention_masks.shape

(100, 1000)

In [7]:
history = workflow_test.fit_offline(
        training_data,
        epochs=2,
        batch_size=256,
        verbose=2,
        # attention_mask=attention_masks
        kwargs={'attention_mask': attention_masks},
    )

INFO:bayesflow:Fitting on dataset instance of OfflineDataset.
INFO:bayesflow:Building on a test batch.


Epoch 1/2
1/1 - 1s - 989ms/step - loss: 1.2305
Epoch 2/2
1/1 - 0s - 255ms/step - loss: 0.6070


INFO:bayesflow:Training completed in 2.22 seconds.


In [8]:
# Load test data
test_data_npz = dict(np.load('./projection_test_set_multistream_odisseo_triaxial.npz', allow_pickle=True))

test_data = {k: np.expand_dims(np.array(test_data_npz[k]), axis=-1) for k in test_data_npz.keys() if k not in ['sim_data']}
test_data['sim_data'] = test_data_npz['sim_data']

print(f"Test sim_data shape: {test_data['sim_data'].shape}")
# Expected: (N_TEST, 2, 1000, 6) -> (n_datasets, n_subjects, n_particles, n_features)

N_TEST = test_data['sim_data'].shape[0]
N_SUBJECTS = test_data['sim_data'].shape[1]  # 2
N_PARTICLES = test_data['sim_data'].shape[2]  # 1000

# Create random attention masks with variable number of valid particles
# Shape needs to be (N_TEST * N_SUBJECTS, N_PARTICLES) - FLATTENED to match how compositional_sample processes data
np.random.seed(42)

min_valid_fraction = 0.5
max_valid_fraction = 1.0

# Create mask with shape (N_TEST, N_SUBJECTS, N_PARTICLES) first
attention_mask_3d = np.zeros((N_TEST, N_SUBJECTS, N_PARTICLES), dtype=np.float32)

for i in range(N_TEST):
    for j in range(N_SUBJECTS):
        valid_fraction = np.random.uniform(min_valid_fraction, max_valid_fraction)
        n_valid = int(N_PARTICLES * valid_fraction)
        valid_indices = np.random.choice(N_PARTICLES, size=n_valid, replace=False)
        attention_mask_3d[i, j, valid_indices] = 1.0

# Flatten to (N_TEST * N_SUBJECTS, N_PARTICLES) to match flattened sim_data
attention_mask_test = attention_mask_3d.reshape(N_TEST * N_SUBJECTS, N_PARTICLES)

print(f"Original mask shape (3D): {attention_mask_3d.shape}")
print(f"Flattened mask shape: {attention_mask_test.shape}")
print(f"Example valid counts (first 5 flattened samples): {attention_mask_test[:5].sum(axis=1)}")

Test sim_data shape: (100, 2, 1000, 6)
Original mask shape (3D): (100, 2, 1000)
Flattened mask shape: (200, 1000)
Example valid counts (first 5 flattened samples): [687. 994. 689. 685. 818.]


In [12]:
# Now use in compositional_sample
N_SAMPLES = 100

def prior_global_score(x: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    return {
        "m_nfw": np.zeros_like(x["m_nfw"]),
        "r_s": np.zeros_like(x["r_s"]),
        "q1": np.zeros_like(x["q1"]),
        "q2": np.zeros_like(x["q2"]),
    }

attention_mask_flat = attention_mask_3d.reshape(N_TEST * N_SUBJECTS, N_PARTICLES)

# Pass attention_mask inside conditions dict instead of as separate kwarg
# This avoids the tensor/non-tensor mixing issue
posterior_batch = workflow_test.compositional_sample(
    num_samples=N_SAMPLES,
    conditions={
        'sim_data': test_data['sim_data'], 
        'j': test_data['j'],
    },
    compute_prior_score=prior_global_score,
    compositional_bridge_d1=1/N_SUBJECTS,
    mini_batch_size=2,
    method='two_step_adaptive',
    steps='adaptive',
    max_steps=1000,
    attention_mask=attention_mask_flat, # Include in conditions with original 3D shape
)

print("Posterior shapes:")
for k, v in posterior_batch.items():
    print(f"  {k}: {v.shape}")

ValueError: In a nested call() argument, you cannot mix tensors and non-tensors. Received invalid mixed argument: kwargs={'compositional_bridge_d1': 0.5, 'mini_batch_size': 2, 'method': 'two_step_adaptive', 'steps': 'adaptive', 'max_steps': 1000, 'attention_mask': tensor([[1., 0., 1.,  ..., 1., 0., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [0., 0., 0.,  ..., 0., 0., 1.],
        ...,
        [0., 1., 0.,  ..., 1., 0., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 0.,  ..., 0., 1., 1.]], device='cuda:0')}